# RF–EBB session review: coverage >90%

List every saved rectangular EBB session whose bearing × distance occupancy grid is more than 90% visited. The threshold is strict (>0.90); it is not an EBB-stability or significance threshold. Use HD class 3 and saved ON RF, with no ADN restriction. Missing EBB bundles and RF pairs are never invented or generated.

Each compatible saved regular RF session is listed separately. RF, HD and EBB must share the Kilosort source. Existing poster dates retain their established HD reference; additional dates use the stated baseline HD session. All HD references are explicit below. These are source choices, not preset cell counts or statistics. Every resulting cohort is selected from saved unit IDs, classifications and RF detections.

Heatmaps use the all-height 0–200 ms RF projection and mean finite EBB map bins, normalized 0 to each row maximum. Each row is one cell, ordered by RF peak. Pairwise uses the existing TC comparison method and up to 10,000 whole-cell permutations (exact enumeration for small cohorts), for all saved distances and 0–8, 8–16, >16 cm. Repeated RF/EBB pairings are separate views of cells, not independent pooled samples. At least three paired cells are required for the existing statistical method.

In [ ]:
from pathlib import Path
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from Utils.direction_comparison import normalize_tc
from Utils.tc_comparison import calculate_tc_statistics
from Utils.plotting import LIGHT_PLOT_STYLE
from scripts.poster_figures import comparison_order
from scripts.poster_rf_ebb import load_sorting_inputs, prepare_sorting_profiles
from scripts.rf_ebb_comparison import source_manifest, pairwise_overview, save_pair_result, BAND_LABELS
from scripts.rf_ebb_session_review import discover_session_pairs, sorting_review_figure, save_review_figure
%matplotlib inline
root = Path(os.environ.get('RFMAPPING_RECORDING_ROOT', '/mnt/senzailab/Kai/#Recording'))
output_dir = Path(os.environ.get('RF_EBB_REVIEW_OUTPUT_DIR', 'rf_ebb_comparison_output/session_review_coverage90'))
output_dir.mkdir(parents=True, exist_ok=True)
minimum_coverage = .90
hd_sessions = {
    ('m19','260827'):9, ('m19','260831'):2,
    ('m20','260918'):9, ('m20','260921'):9,
    ('m21','261006'):9,
    ('m22','261008'):9, ('m22','261009'):4, ('m22','261015'):3, ('m22','261019'):3,
    ('m23','261015'):9, ('m23','261019'):3,
    ('m24','261103'):10, ('m24','261104'):10,
}
n_permutations, random_seed = 10000, 1
plt.rcParams.update({**LIGHT_PLOT_STYLE, 'figure.facecolor':'white','axes.facecolor':'white',
    'savefig.facecolor':'white','savefig.transparent':False,'svg.fonttype':'none'})

## Discover qualified sessions and all saved RF matches

In [ ]:
inventory, specifications = discover_session_pairs(root, hd_sessions, minimum_coverage=minimum_coverage)
(output_dir / 'session_inventory.json').write_text(json.dumps(inventory, indent=2) + '\n')
session_table = pd.DataFrame(inventory)
session_table.to_csv(output_dir / 'session_inventory.csv', index=False)
display(session_table[['mouse','date','ebb','hd','coverage','hd_class3','rf_pairs','status']])
print(len(specifications), 'RF–EBB pairings ready for review')

## Generate each paired comparison from recordings

In [ ]:
records, manifests = [], []
for spec in specifications:
    stem = f"{spec['mouse']}_{spec['date']}_ebb{spec['ebb']}_rf{spec['rf']}_hd{spec['hd']}_{spec['probe']}"
    folder = output_dir / stem
    folder.mkdir(exist_ok=True)
    loaded = load_sorting_inputs(root, spec)
    prepared = prepare_sorting_profiles(loaded)
    metadata = prepared['metadata']
    assert metadata['coverage'] > minimum_coverage
    manifests.append(dict(name=stem, **source_manifest(loaded), quality=metadata))
    profiles = {key: normalize_tc(table) for key, table in prepared['profiles'].items()}
    order = comparison_order(profiles['RF'])
    label = f"{spec['mouse'].upper()} {spec['date']} · RF S{spec['rf']} / EBB S{spec['ebb']} · HD S{spec['hd']} · Probe {spec['probe']}"
    fig = sorting_review_figure(profiles, order, label=label, coverage=metadata['coverage'])
    save_review_figure(fig, folder, 'rf_ebb_sorted')
    statistics = {}
    for band in BAND_LABELS:
        stats = calculate_tc_statistics(profiles['RF'], profiles[band], bins=15,
            n_permutations=n_permutations, random_seed=random_seed)
        statistics[band] = stats
        save_pair_result(folder, 'statistics', band, profiles['RF'], profiles[band], stats, [])
    fig = pairwise_overview(statistics, title=label)
    save_review_figure(fig, folder, 'pairwise_bands')
    fig = pairwise_overview({'all':statistics['all']}, title=label)
    save_review_figure(fig, folder, 'pairwise_all')
    prepared['metrics'].to_csv(folder / 'unit_quality.csv')
    prepared['native_rf'].to_csv(folder / 'native_rf_counts.csv')
    row = dict(name=stem, **spec, coverage=metadata['coverage'], cells=len(order),
        unique_pairs=len(order)*(len(order)-1)//2, median_half_r=metadata['median_half_r'],
        held_fraction=metadata['held_fraction'], **statistics['all']['direction']['pairwise'])
    records.append(row)
    print(stem, len(order), 'cells; coverage', round(100*metadata['coverage'],2), flush=True)
    del loaded, prepared
review_table = pd.DataFrame(records)
review_table.to_csv(output_dir / 'review_summary.csv', index=False)
(output_dir / 'sources.json').write_text(json.dumps(manifests, indent=2) + '\n')
display(review_table[['mouse','date','ebb','rf','hd','coverage','cells','spearman_mantel_rho','permutation_p','median_half_r','held_fraction']])

## Visual index by mouse

Each row contains the five sorting panels on the left and the all-distance pairwise result on the right. The original SVG/PNG files and four-band pairwise figures remain available in each named folder.

In [ ]:
def save_contact_sheet(rows, filename, *, show=False):
    fig, axes = plt.subplots(len(rows),2,figsize=(18,5*len(rows)),
        gridspec_kw={'width_ratios':[2.65,1]},squeeze=False,layout='constrained',facecolor='white')
    for axes_row, record in zip(axes, rows.itertuples()):
        for axis, image_name in zip(axes_row, ['rf_ebb_sorted','pairwise_all']):
            axis.imshow(plt.imread(output_dir / record.name / f'{image_name}.png'))
            axis.axis('off')
    fig.savefig(output_dir / filename,dpi=140,facecolor='white',transparent=False)
    if show:
        display(fig)
    plt.close(fig)

for (mouse, date, ebb), rows in review_table.groupby(['mouse','date','ebb'],sort=True):
    save_contact_sheet(rows, f'{mouse}_{date}_ebb{ebb}_review.png')
for mouse, rows in review_table.groupby('mouse',sort=True):
    for page, start in enumerate(range(0,len(rows),4),start=1):
        save_contact_sheet(rows.iloc[start:start+4], f'{mouse}_review_{page}.png', show=True)
print('All session comparisons exported. No Illustrator document was modified.')